# Solar Filament Segmentation Challenge 2026 — reproduction notebook

This notebook runs the **entire pipeline of this repository** end to end, without any training:

1. locate the data and load `models/best_model.pth` (strict `state_dict` load),
2. build the 3-channel input and inspect a single test image,
3. run full-disk inference (4-view flip TTA) on **all test images** and write `submission.csv`,
4. validate the submission file (structure, RLE decoding, no overlapping instances),
5. evaluate on the held-out validation split (Dice, Panoptic Quality, fragmentation).

All logic lives in importable modules (`model.py`, `data.py`, `pipeline.py`); the notebook only calls them.

**Scope note.** This repository contains the segmentation model and its inference/evaluation code. It does
*not* regenerate the Kaggle leaderboard submission `58173877`, which was assembled by a separate multi-stage
process (see `README.md`, section *Relationship to the leaderboard submission*). Numbers shown here come from
the cells below, nothing is hard-coded.

**Data.** Download the competition data from Kaggle and either place it in `data/MAGFiLO_1.0_Kaggle_2026/`
(containing `train/` and `test/`) or set the environment variable `FILAMENT_DATA_ROOT`. On Kaggle the data is
found automatically.

In [ ]:
import os, sys, time
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm

sys.path.insert(0, ".")  # make the repo modules importable when the notebook is run from the repo root
from data import list_images, load_gt_rles, load_instances_by_stem, split_stems
from model import make_3channel_image
from pipeline import (InferenceConfig, evaluate, extract_instances, load_trained_model, predict_probability,
                      print_checks, rles_from_masks, run_inference, validate_submission)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device, "| torch", torch.__version__, "| opencv", cv2.__version__)

In [ ]:
candidates = [os.environ.get("FILAMENT_DATA_ROOT"),
              "data/MAGFiLO_1.0_Kaggle_2026",
              "/kaggle/input/competitions/filament-segmentation-2026/MAGFiLO_1.0_Kaggle_2026",
              "/kaggle/input/filament-segmentation-2026/MAGFiLO_1.0_Kaggle_2026"]
ROOT = next((Path(c) for c in candidates if c and Path(c).exists()), None)
assert ROOT is not None, "Data not found: put MAGFiLO_1.0_Kaggle_2026/ under data/ or set FILAMENT_DATA_ROOT"

TEST_DIR  = ROOT / "test" / "test_images"
TRAIN_IMG = ROOT / "train" / "train_images"
TRAIN_ANN = ROOT / "train" / "MAGFiLO_1.0_Annotations_kaggle2026_train.json"
WEIGHTS   = Path("models/best_model.pth")
for p in (TEST_DIR, WEIGHTS):
    assert p.exists(), f"missing: {p}"
print("data root:", ROOT.resolve())
print("test images:", len(list_images(TEST_DIR)), "| train annotations available:", TRAIN_ANN.exists())

## 1. Load the trained model

`FilamentResNet34UNet` (ResNet-34 encoder, transposed-convolution decoder, 24.45 M parameters) is loaded with
`strict=True`, so any mismatch between code and checkpoint raises an error. The stored `best_val_dice` is the
crop-level Dice measured during training (crops centred on annotated filaments), **not** a full-disk score.

In [ ]:
model, meta = load_trained_model(WEIGHTS, device)
print("checkpoint metadata:", meta)
print(f"parameters: {sum(p.numel() for p in model.parameters()):,}")

## 2. Input representation and a single prediction

Three channels, all in [0, 1]: raw H-alpha intensity, CLAHE (clip 2.0, 8x8 tiles) and Sobel gradient magnitude
(normalised by its maximum). The image is resized to 1024x1024 for the network, predictions of 4 flip views are
averaged, and the probability map is resized back to 2048x2048.

In [ ]:
sample_path = list_images(TEST_DIR)[0]
gray = cv2.imread(str(sample_path), cv2.IMREAD_GRAYSCALE)
print(sample_path.name, gray.shape, gray.dtype)

x3 = make_3channel_image(cv2.resize(gray, (1024, 1024), interpolation=cv2.INTER_AREA))
prob = predict_probability(model, gray, device, infer_size=1024, tta=4)
cfg0 = InferenceConfig()  # default configuration (selected on the validation split)
instances = extract_instances(prob, cfg0.threshold, cfg0.min_area)

overlay = cv2.cvtColor(gray, cv2.COLOR_GRAY2RGB)
rng = np.random.default_rng(0)
for m in instances:
    overlay[m > 0] = (0.5 * overlay[m > 0] + 0.5 * rng.integers(80, 255, 3)).astype(np.uint8)

fig, ax = plt.subplots(1, 4, figsize=(20, 5.5))
for i, name in enumerate(["raw", "CLAHE", "Sobel"]):
    ax[i].imshow(x3[..., i], cmap="gray"); ax[i].set_title(f"channel {i}: {name}"); ax[i].axis("off")
ax[3].imshow(overlay); ax[3].set_title(f"{len(instances)} predicted instances"); ax[3].axis("off")
plt.tight_layout(); plt.show()

## 3. Inference on the whole test set -> `submission.csv`

Default configuration (`InferenceConfig`, selected on the validation split, see `figures/val_metrics.json`):
1024x1024 network input, 4-view flip TTA, probability threshold 0.98, 8-connected components as instances,
components smaller than 1000 px (in the 2048x2048 frame) discarded. Each row is
`<image_stem>_<k>` with the compressed COCO RLE counts of one instance.

In [ ]:
cfg = InferenceConfig()  # infer_size=1024, tta=4, threshold=0.98, min_area=1000, opening=False
test_images = list_images(TEST_DIR)

t0 = time.time()
submission = run_inference(model, test_images, device, cfg, progress=lambda xs: tqdm(xs, unit="img"))
elapsed = time.time() - t0
submission.to_csv("submission.csv", index=False)
print(f"{len(submission)} instances in {submission['filament_id'].str.rsplit('_', n=1).str[0].nunique()} images "
      f"| {elapsed / len(test_images):.2f} s/image on {device} | config={cfg}")
submission.head()

## 4. Validate the submission file

Ten structural checks (columns, non-empty, no null RLE, unique ids, id format, no quote characters, every RLE
decodes at 2048x2048, no empty masks, no pixel shared by two instances of an image, coverage of the test images).

In [ ]:
ok = print_checks(validate_submission(pd.read_csv("submission.csv"), expected_stems=[p.stem for p in test_images]))
print("ALL CHECKS PASSED" if ok else "SOME CHECKS FAILED")

## 5. Held-out validation (Dice / Panoptic Quality)

The model was trained on the first 80 % of the (sorted) training images; the last 20 % (141 images) are the
validation set and were never used for training or checkpoint selection beyond the crop-level Dice. This cell
needs the competition *train* data. `evaluate` is our implementation of the Panoptic Quality formula
(IoU > 0.5, one-to-one matching, pooled over images) and may differ in details from the official scorer.

In [ ]:
if TRAIN_ANN.exists() and TRAIN_IMG.exists():
    by_stem = load_instances_by_stem(TRAIN_ANN, TRAIN_IMG)
    _, val_stems = split_stems(list(by_stem))
    gt_all = load_gt_rles(TRAIN_ANN)
    gt = {s: gt_all[s] for s in val_stems}

    # One forward pass per image; every post-processing variant is RLE-encoded immediately so that only
    # compact RLEs (not full-frame probability maps / masks) are kept in memory.
    variants = [(thr, opening) for thr in (0.5, 0.9, 0.98) for opening in (False, True)]
    preds = {v: {} for v in variants}
    for s in tqdm(val_stems, unit="img"):
        g = cv2.imread(str(by_stem[s][0]["image_path"]), cv2.IMREAD_GRAYSCALE)
        prob = predict_probability(model, g, device, cfg.infer_size, cfg.tta)
        for thr, opening in variants:
            preds[(thr, opening)][s] = rles_from_masks({s: extract_instances(prob, thr, cfg.min_area, opening)})[s]

    rows = []
    for (thr, opening), p in preds.items():
        r = evaluate(gt, p, val_stems)
        rows.append({"threshold": thr, "opening": opening, **{k: round(v, 4) if isinstance(v, float) else v for k, v in r.items() if k != "n_images"}})
    results = pd.DataFrame(rows)
    display(results)
else:
    print("Train annotations/images not found - skipping validation (inference above does not need them).")

### Reading the numbers

* `mean_dice` is the per-image pixel Dice between the union of ground-truth and predicted filaments.
* `PQ = SQ x RQ` (Panoptic Quality) is dominated by recognition quality: each filament must be detected as
  *one* object with IoU > 0.5. `one_to_many` counts ground-truth filaments split into several predictions and
  `many_to_one` counts predictions that merge several ground-truth filaments (our definition, see
  `pipeline.evaluate`).
* The network is trained on single-instance crops and applied to the full disk, so pixel Dice is clearly higher
  than instance-level PQ; the instance separation step is plain connected-component labelling.